# Qwen3.8-27B: predictions and self-verification

Runs the study's prompts (`data/prompts`, the same files as the GPT-6 Luna runs) through
Qwen3.8-27B in bf16 with Hugging Face Transformers on one 80 GB GPU, at low reasoning, and then
has the same model, with the same settings, judge its own answers (self-verification).

Results are written in the repository's formats and at its paths (`data/predictions`,
`data/prompts/verification`, `data/verifications`), so the results zip made at the end unpacks
into the repository root and the usual scripts read it unchanged. The pilot, the log-probability
of every generated token and a session log go to `_local/qwen38_27b/`.

Order: environment, checks, pilot (10 prompts per dataset and their self-verification), full
predictions, self-verification, results zip. Every run resumes: after an interruption or a
session time limit, rerun from the top and saved items are skipped.

Model: https://huggingface.co/Qwen/Qwen3.8-27B

## 1. Environment

Run the first code cell once from the working cluster AI kernel. It installs a private package
overlay beside this notebook and registers a new Jupyter kernel. It does not modify the
read-only cluster installation, install another PyTorch build, or require a terminal.

After it finishes, refresh JupyterLab, select `Python (Qwen3.8 Overlay)`, restart the kernel,
and run from the top.

In [ ]:
# Run this cell once from the working cluster AI kernel. It intentionally uses
# that kernel's CUDA-compatible PyTorch and Triton installations.
import json
import os
import subprocess
import sys
from pathlib import Path

OVERLAY_DIR = Path.cwd().resolve() / ".qwen38_python_overlay"
ACTIVE_OVERLAY = os.environ.get("QWEN38_PYTHON_OVERLAY")

if ACTIVE_OVERLAY == str(OVERLAY_DIR):
    print("Qwen3.8 package overlay is active:", ACTIVE_OVERLAY)
else:
    print("Base Python:", sys.executable)
    print("Installing private overlay at:", OVERLAY_DIR)
    subprocess.check_call([
        sys.executable,
        "-c",
        (
            "import torch, triton; "
            "print('torch', torch.__version__, torch.__file__); "
            "print('triton', triton.__version__, triton.__file__)"
        ),
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--target",
        str(OVERLAY_DIR),
        "--upgrade",
        "--no-deps",
        "einops==0.8.2",
        "fla-core==0.5.2",
        "tokenizers==0.22.2",
        "transformers==5.13.1",
        "flash-linear-attention==0.5.2",
    ])

    from jupyter_core.paths import jupyter_data_dir

    kernel_dir = (
        Path(jupyter_data_dir()) / "kernels" / "qwen38-transformers-overlay"
    )
    kernel_dir.mkdir(parents=True, exist_ok=True)
    kernel_spec = {
        "argv": [
            sys.executable,
            "-m",
            "ipykernel_launcher",
            "-f",
            "{connection_file}",
        ],
        "display_name": "Python (Qwen3.8 Overlay)",
        "language": "python",
        "env": {
            "PYTHONPATH": str(OVERLAY_DIR),
            "QWEN38_PYTHON_OVERLAY": str(OVERLAY_DIR),
        },
    }
    (kernel_dir / "kernel.json").write_text(
        json.dumps(kernel_spec, indent=2) + "\n",
        encoding="utf-8",
    )
    print("Registered kernel:", kernel_dir)
    print(
        "Refresh JupyterLab, select Python (Qwen3.8 Overlay), restart the "
        "kernel, and rerun from the first cell."
    )

In [ ]:
import gc
import importlib.metadata
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
import transformers
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    LogitsProcessor,
    LogitsProcessorList,
)
from transformers.utils import is_flash_linear_attention_available

print("Python:", sys.version)
print("Python executable:", sys.executable)
# Versions that can change the outputs; a resumed run must use the same ones.
PACKAGE_VERSIONS = {}
for package in ("torch", "triton", "transformers", "tokenizers", "accelerate", "flash-linear-attention", "fla-core"):
    try:
        PACKAGE_VERSIONS[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        PACKAGE_VERSIONS[package] = None
    print(f"{package}: {PACKAGE_VERSIONS[package] or 'NOT INSTALLED'}")

print("PyTorch CUDA build:", torch.version.cuda)
print("Transformers module:", transformers.__file__)
print("CUDA available:", torch.cuda.is_available())
ACTIVE_OVERLAY = os.environ.get("QWEN38_PYTHON_OVERLAY")
if not ACTIVE_OVERLAY:
    raise RuntimeError(
        "The private Qwen package overlay is not active. Run the setup cell, "
        "refresh JupyterLab, and select Python (Qwen3.8 Overlay)."
    )
if not Path(transformers.__file__).resolve().is_relative_to(
    Path(ACTIVE_OVERLAY).resolve()
):
    raise RuntimeError(
        "Transformers was not imported from the private Qwen package overlay: "
        f"{transformers.__file__}"
    )
if not torch.cuda.is_available():
    raise RuntimeError(
        "This kernel cannot see a GPU. Start a GPU-backed Jupyter session and "
        "select the Transformers kernel."
    )
if not is_flash_linear_attention_available():
    raise RuntimeError(
        "Flash Linear Attention is unavailable. Install the pinned "
        "flash-linear-attention[cuda] package, restart the kernel, and rerun "
        "this cell. Do not continue with the memory-heavy PyTorch fallback."
    )

### Transformers fix: chunked prefill

Transformers 5.13.1 cuts the position ids of each prefill chunk with `[:, start:end]`. Qwen3.5's
position ids are `[4, batch, length]`, so this slices the batch axis and every chunk after the
first 2,048 tokens fails ("Sizes of tensors must match ... Expected size 0 but got size 1").
The cell below patches that one line in memory to slice the sequence axis. Prompts of up to
2,048 tokens are prefilled in one chunk and are unaffected either way.

In [ ]:
import inspect
import textwrap

import transformers.generation.utils as generation_utils

if not getattr(generation_utils.GenerationMixin._prefill, "slices_sequence_axis", False):
    chunk_slice = "position_ids[:, past_length:current_length]"
    source = textwrap.dedent(inspect.getsource(generation_utils.GenerationMixin._prefill))
    if source.count(chunk_slice) != 1:
        raise RuntimeError("GenerationMixin._prefill has changed; check this patch against the installed Transformers")
    patched = {}
    exec(compile(source.replace(chunk_slice, "position_ids[..., past_length:current_length]"),
                 "<patched GenerationMixin._prefill>", "exec"), vars(generation_utils), patched)
    patched["_prefill"].slices_sequence_axis = True
    generation_utils.GenerationMixin._prefill = patched["_prefill"]
print("Chunked prefill slices position ids on the sequence axis")

## 2. Configuration

The study runs models other than GPT-6 Luna at low reasoning only. `CONDITION = "none"` also
works here and writes a separate run. `ADDITIONS = True` runs only the ten RE2 items added once
RE2 kept everything but its CRUXEval samples (`data/prompts/re2_additions/`), as a supplementary
run that the analysis combines with the main one; it skips the pilot, which would be the same items.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
from statistics import fmean, median
import sys
import zipfile

BUNDLE_ROOT = Path.cwd().resolve()
sys.path[:0] = [str(BUNDLE_ROOT / "src"), str(BUNDLE_ROOT / "scripts")]
from datasets import sha256_file
from generate_verification_prompts import generate as generate_verification_prompts
from run_predictions import load_prompts, now, read_results
from runs import load_run
from scoring import parse_confidence, parse_prediction, parse_verdict, prediction_logprobs, verdict_probability

MODEL = {
    "model_id": "Qwen/Qwen3.8-27B",
    "revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
    "precision": "bfloat16",
}
PROVIDER = "local_transformers"

# Qwen3.8 has a native 262,144-token context; the audit below checks every prompt fits.
MAX_TOTAL_TOKENS = 262_144
MAX_NEW_TOKENS = 16_384  # as in the GPT-6 Luna runs
PREFILL_CHUNK_SIZE = 2_048

# Sampling, applied by the recorder below after it takes the raw-model log-probability.
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
REPETITION_PENALTY = 1.0
SEED = 0

CONDITIONS = {
    "none": {"enable_thinking": False, "reasoning_effort": None, "preserve_thinking": False},
    "low": {"enable_thinking": True, "reasoning_effort": "low", "preserve_thinking": False},
}
CONDITION = "low"
ADDITIONS = True  # the ten added RE2 items only; False for the full study prompts
RUN_NAME = {"none": "qwen3.8_27b_no_reasoning", "low": "qwen3.8_27b_low_reasoning"}[CONDITION]
RUN_NAME += "_re2_additions" if ADDITIONS else ""
VERIFICATION_RUN = RUN_NAME + "_self_verification"

DATASETS = ("re2",) if ADDITIONS else ("re2", "cruxeval")  # RE2 first: it carries the comparison across models
EXPECTED_COUNTS = {"re2": 10} if ADDITIONS else {"re2": 456, "cruxeval": 800}
DATA = BUNDLE_ROOT / "data"
PROMPT_DIR = DATA / "prompts" / "re2_additions" if ADDITIONS else DATA / "prompts"
EXTRA = BUNDLE_ROOT / "_local" / "qwen38_27b"  # pilot, token-level log-probabilities, session log
PILOT = EXTRA / "pilot"
PILOT_SIZE = 10
RETRY_ERRORS = False  # True retries requests saved as errors (e.g. out of GPU memory)


def template_kwargs(condition):
    profile = CONDITIONS[condition]
    kwargs = {"enable_thinking": profile["enable_thinking"], "preserve_thinking": profile["preserve_thinking"]}
    if profile["reasoning_effort"] is not None:
        kwargs["reasoning_effort"] = profile["reasoning_effort"]
    return kwargs


print("Bundle:", BUNDLE_ROOT)
print("Runs:", RUN_NAME, "and", VERIFICATION_RUN)

## 3. Bundle, hardware and prompts

In [ ]:
bundle = json.loads((BUNDLE_ROOT / "bundle_manifest.json").read_text(encoding="utf-8"))
changed = [path for path, digest in bundle["files"].items() if sha256_file(BUNDLE_ROOT / path) != digest]
if changed:
    raise RuntimeError(f"Files differ from bundle_manifest.json: {changed}")
print(f"{len(bundle['files'])} files verified; bundle built {bundle['created_at']} from commit {bundle['git_commit']}")

In [ ]:
def run_checked(command):
    return subprocess.run(command, check=True, text=True, capture_output=True).stdout.strip()


GPU_INVENTORY = run_checked([
    "nvidia-smi",
    "--query-gpu=index,name,memory.total,driver_version",
    "--format=csv,noheader",
]).splitlines()
print("\n".join(GPU_INVENTORY))
print("Disk free (GiB):", round(shutil.disk_usage(BUNDLE_ROOT).free / 2**30, 1))
EXTRA.mkdir(parents=True, exist_ok=True)
with (EXTRA / "sessions.jsonl").open("a", encoding="utf-8") as stream:
    stream.write(json.dumps({"started_at": now(), "bundle_commit": bundle["git_commit"], "gpu": GPU_INVENTORY,
                             "python": sys.version, "platform": platform.platform(),
                             "packages": PACKAGE_VERSIONS}) + "\n")

In [ ]:
def prompt_paths(directory):
    """The directory's prompt files, in DATASETS order."""
    return [directory / f"{name}_prompts.jsonl" for name in DATASETS if (directory / f"{name}_prompts.jsonl").exists()]


# The runner's own loader: one user message per prompt, whose hash must match.
PROMPTS = load_prompts(prompt_paths(PROMPT_DIR), None)
counts = Counter(prompt["dataset"] for prompt in PROMPTS)
if counts != Counter(EXPECTED_COUNTS):
    raise RuntimeError(f"Expected {EXPECTED_COUNTS} prompts, found {dict(counts)}")
for path in prompt_paths(PROMPT_DIR):
    print(path.name, counts[path.name.removesuffix("_prompts.jsonl")], "prompts, sha256", sha256_file(path))

## 4. Load the processor and pinned bf16 model

The model is forced onto GPU 0. On a genuine 80 GB allocation, this avoids silent CPU
offloading. Loading may take several minutes on the first run while the 55.6 GB checkpoint is
downloaded.

In [ ]:
PROCESSOR = AutoProcessor.from_pretrained(
    MODEL["model_id"],
    revision=MODEL["revision"],
    trust_remote_code=False,
)
TOKENIZER = PROCESSOR.tokenizer

torch.cuda.empty_cache()
MODEL_INSTANCE = AutoModelForMultimodalLM.from_pretrained(
    MODEL["model_id"],
    revision=MODEL["revision"],
    dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    trust_remote_code=False,
)
MODEL_INSTANCE.eval()
MODEL_DEVICE = next(MODEL_INSTANCE.parameters()).device

# Import availability is not enough: Qwen silently falls back to a much more
# memory-hungry pure-PyTorch Gated DeltaNet implementation when the optimized
# function was not wired into the loaded layers. Inspect the actual callable
# selected by every linear-attention layer before attempting a long prompt.
LINEAR_ATTENTION_BACKENDS = sorted({
    (
        f"{getattr(module.chunk_gated_delta_rule, '__module__', '')}."
        f"{getattr(module.chunk_gated_delta_rule, '__name__', type(module.chunk_gated_delta_rule).__name__)}"
    )
    for module in MODEL_INSTANCE.modules()
    if hasattr(module, "chunk_gated_delta_rule")
})
print("Gated DeltaNet backends:", LINEAR_ATTENTION_BACKENDS)
if not LINEAR_ATTENTION_BACKENDS:
    raise RuntimeError("Could not find the model's Gated DeltaNet backend")
if any(
    "torch_chunk_gated_delta_rule" in backend
    for backend in LINEAR_ATTENTION_BACKENDS
):
    raise RuntimeError(
        "The loaded model is still using Transformers' memory-heavy pure-"
        "PyTorch Gated DeltaNet fallback. Restart the kernel after installing "
        "flash-linear-attention[cuda]==0.5.2, then rerun from the first cell."
    )

PARAMETER_DEVICES = Counter(
    str(parameter.device) for parameter in MODEL_INSTANCE.parameters()
)
print("Parameter devices:", PARAMETER_DEVICES)
print("Model dtype:", next(MODEL_INSTANCE.parameters()).dtype)
print(
    "GPU allocated/reserved (GiB):",
    round(torch.cuda.memory_allocated() / 2**30, 2),
    round(torch.cuda.memory_reserved() / 2**30, 2),
)
if any(device != "cuda:0" for device in PARAMETER_DEVICES):
    raise RuntimeError(f"Model was not loaded fully on cuda:0: {PARAMETER_DEVICES}")

## 5. Exact prompt lengths

In [ ]:
def encode_messages(messages, condition, *, return_tensors):
    return PROCESSOR.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors=return_tensors,
        **template_kwargs(condition),
    )


def audit(prompts, condition):
    """Exact prompt lengths under the chat template; stops if a prompt and the output budget
    exceed the context. Returns the longest prompt's length and the prompt."""
    longest = None
    for index, prompt in enumerate(prompts, start=1):
        token_ids = encode_messages(prompt["messages"], condition, return_tensors=None)["input_ids"]
        if token_ids and isinstance(token_ids[0], list):
            token_ids = token_ids[0]
        if len(token_ids) + MAX_NEW_TOKENS > MAX_TOTAL_TOKENS:
            raise RuntimeError(f"{prompt['dataset']} {prompt['id']}: {len(token_ids)} prompt tokens leave no room "
                               f"for {MAX_NEW_TOKENS} generated tokens")
        if longest is None or len(token_ids) > longest[0]:
            longest = (len(token_ids), prompt)
        if index % 250 == 0:
            print("Audited", index, "prompts")
    print("Longest prompt:", longest[0], "tokens,", longest[1]["dataset"], longest[1]["id"])
    return longest


LONGEST_TOKENS, LONGEST_PROMPT = audit(PROMPTS, CONDITION)

## 6. Generation and token log-probabilities

`generate()` normally retains a full vocabulary score vector for every token when
`output_scores=True`, which is unnecessarily memory intensive. The custom processor below
retains only one step at a time. It records the selected token's log-probability under the raw
model distribution, then applies the frozen temperature, top-k, top-p and min-p filters.

Each completion is returned in the runner's format: the visible answer as `text`, the thinking
as `reasoning`, token usage, and the log-probabilities of the answer's tokens in OpenAI's layout
(token, bytes, logprob), which is what the repository's evaluation reads. Batch size is one.

In [ ]:
class SamplingAndLogprobRecorder(LogitsProcessor):
    def __init__(
        self,
        *,
        temperature: float,
        top_k: int,
        top_p: float,
        min_p: float,
    ):
        self.temperature = temperature
        self.top_k = top_k
        self.top_p = top_p
        self.min_p = min_p
        self._previous_raw_scores = None
        self.token_logprobs = []

    def _record_selected(
        self,
        selected_token_ids: torch.LongTensor,
    ) -> None:
        if self._previous_raw_scores is None:
            return
        raw = self._previous_raw_scores.float()
        selected_logits = raw.gather(
            dim=-1,
            index=selected_token_ids.unsqueeze(-1),
        ).squeeze(-1)
        selected_logprobs = selected_logits - torch.logsumexp(raw, dim=-1)
        if selected_logprobs.numel() != 1:
            raise RuntimeError("Token-logprob recorder requires batch size one")
        self.token_logprobs.append(float(selected_logprobs.item()))

    def __call__(
        self,
        input_ids: torch.LongTensor,
        scores: torch.FloatTensor,
    ) -> torch.FloatTensor:
        if self._previous_raw_scores is not None:
            self._record_selected(input_ids[:, -1])
        self._previous_raw_scores = scores.detach().clone()

        warped = scores
        if self.temperature <= 0:
            raise ValueError("Temperature must be positive")
        if self.temperature != 1.0:
            warped = warped / self.temperature

        if self.top_k > 0:
            keep = min(self.top_k, warped.shape[-1])
            top_values, top_indices = torch.topk(warped, keep, dim=-1)
            if 0.0 < self.top_p < 1.0:
                cumulative = torch.softmax(
                    top_values.float(),
                    dim=-1,
                ).cumsum(dim=-1)
                remove_top = cumulative > self.top_p
                remove_top[:, 1:] = remove_top[:, :-1].clone()
                remove_top[:, 0] = False
                top_values = top_values.masked_fill(remove_top, -math.inf)
            filtered = torch.full_like(warped, -math.inf)
            warped = filtered.scatter(
                dim=-1,
                index=top_indices,
                src=top_values,
            )
        elif 0.0 < self.top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(
                warped,
                descending=True,
                dim=-1,
            )
            cumulative = torch.softmax(
                sorted_logits.float(),
                dim=-1,
            ).cumsum(dim=-1)
            sorted_remove = cumulative > self.top_p
            sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
            sorted_remove[:, 0] = False
            remove = torch.zeros_like(sorted_remove).scatter(
                dim=-1,
                index=sorted_indices,
                src=sorted_remove,
            )
            warped = warped.masked_fill(remove, -math.inf)

        if self.min_p > 0.0:
            min_allowed = (
                warped.amax(dim=-1, keepdim=True) + math.log(self.min_p)
            )
            warped = warped.masked_fill(warped < min_allowed, -math.inf)

        return warped

    def finalize(self, sequences: torch.LongTensor) -> list:
        if self._previous_raw_scores is not None:
            self._record_selected(sequences[:, -1])
            self._previous_raw_scores = None
        return self.token_logprobs


def reset_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def release_memory():
    gc.collect()
    torch.cuda.empty_cache()


def eos_token_ids():
    configured = MODEL_INSTANCE.generation_config.eos_token_id
    if configured is None:
        return set()
    if isinstance(configured, int):
        return {configured}
    return {int(token_id) for token_id in configured}


EOS_TOKEN_IDS = eos_token_ids()
SPECIAL_TOKEN_IDS = {int(token_id) for token_id in TOKENIZER.all_special_ids}
CLOSE_THINK_IDS = TOKENIZER.encode("</think>", add_special_tokens=False)
print("EOS token IDs:", EOS_TOKEN_IDS)
print("Closing-think token IDs:", CLOSE_THINK_IDS)


def find_subsequence(values, needle):
    if not needle:
        return None
    for start in range(max(0, len(values) - len(needle) + 1)):
        if values[start : start + len(needle)] == needle:
            return start
    return None


def byte_decoder():
    """Byte-level BPE writes each byte as a printable character (GPT-2's bytes_to_unicode);
    this maps those characters back to bytes."""
    printable = [*range(ord("!"), ord("~") + 1), *range(ord("\u00a1"), ord("\u00ac") + 1),
                 *range(ord("\u00ae"), ord("\u00ff") + 1)]
    others = [byte for byte in range(256) if byte not in printable]
    return {**{chr(byte): byte for byte in printable}, **{chr(256 + i): byte for i, byte in enumerate(others)}}


BYTE_DECODER = byte_decoder()


def token_bytes(token_id):
    """The exact bytes a token adds to the text, so answer log-probabilities align with it."""
    piece = TOKENIZER.convert_ids_to_tokens(token_id)
    if all(character in BYTE_DECODER for character in piece):
        return bytes(BYTE_DECODER[character] for character in piece)
    return TOKENIZER.decode([token_id], skip_special_tokens=False, clean_up_tokenization_spaces=False).encode("utf-8")


PROBE = "def f(x):\n    return {'\u03c0': [1, 2.5e-3], \"t\": None}  # \u6d4b\u8bd5 \u2713\n"
if b"".join(token_bytes(t) for t in TOKENIZER.encode(PROBE, add_special_tokens=False)) != PROBE.encode("utf-8"):
    raise RuntimeError("Token bytes do not reproduce the text, so answer log-probabilities could not be aligned")


def decode(token_ids):
    return TOKENIZER.decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)


@torch.inference_mode()
def generate_completion(messages, condition, *, max_new_tokens=MAX_NEW_TOKENS, seed=SEED):
    """One sampled completion in the runner's format, plus every generated token and its
    log-probability (token_ids, token_logprobs) for the token-level log."""
    reset_seed(seed)
    encoded = encode_messages(messages, condition, return_tensors="pt")
    inputs = {key: value.to(MODEL_DEVICE) if isinstance(value, torch.Tensor) else value
              for key, value in encoded.items()}
    prompt_tokens = int(inputs["input_ids"].shape[-1])
    if prompt_tokens + max_new_tokens > MAX_TOTAL_TOKENS:
        raise RuntimeError(f"Prompt {prompt_tokens} + generation {max_new_tokens} exceeds {MAX_TOTAL_TOKENS}")

    recorder = SamplingAndLogprobRecorder(temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P, min_p=MIN_P)
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    generated = MODEL_INSTANCE.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        # Neutralize checkpoint defaults. The recorder applies the frozen
        # sampling filters after capturing raw-model token probabilities.
        temperature=1.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=REPETITION_PENALTY,
        logits_processor=LogitsProcessorList([recorder]),
        return_dict_in_generate=True,
        output_scores=False,
        use_cache=True,
        prefill_chunk_size=PREFILL_CHUNK_SIZE,
        pad_token_id=TOKENIZER.pad_token_id,
        eos_token_id=sorted(EOS_TOKEN_IDS),
    )
    elapsed = time.perf_counter() - started
    sequences = generated.sequences
    token_ids = [int(token_id) for token_id in sequences[0, prompt_tokens:].tolist()]
    token_logprobs = recorder.finalize(sequences)
    if len(token_ids) != len(token_logprobs):
        raise RuntimeError(f"Token/logprob misalignment: {len(token_ids)} != {len(token_logprobs)}")

    # The thinking ends at the first </think>; the visible answer follows it.
    marker = find_subsequence(token_ids, CLOSE_THINK_IDS) if CONDITIONS[condition]["enable_thinking"] else None
    if not CONDITIONS[condition]["enable_thinking"]:
        response_start = 0
    elif marker is None:
        response_start = len(token_ids)  # truncated while thinking
    else:
        response_start = marker + len(CLOSE_THINK_IDS)
    reasoning = decode(token_ids[:response_start if marker is None else marker]).strip()
    visible = [(token_id, logprob) for token_id, logprob in zip(token_ids[response_start:], token_logprobs[response_start:])
               if token_id not in SPECIAL_TOKEN_IDS]
    result = {
        "text": decode(token_ids[response_start:]),
        "reasoning": reasoning,
        "finish_reason": "stop" if token_ids and token_ids[-1] in EOS_TOKEN_IDS else "length",
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": len(token_ids),
                  "total_tokens": prompt_tokens + len(token_ids),
                  "completion_tokens_details": {"reasoning_tokens": response_start}},
        "logprobs": {"content": [{"token": TOKENIZER.decode([token_id], skip_special_tokens=False,
                                                            clean_up_tokenization_spaces=False),
                                  "bytes": list(token_bytes(token_id)), "logprob": logprob}
                                 for token_id, logprob in visible]},
        "latency_seconds": elapsed,
        "tokens_per_second": len(token_ids) / elapsed if elapsed > 0 else None,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated() / 2**30,
        "token_ids": token_ids,
        "token_logprobs": token_logprobs,
    }
    del generated, sequences, inputs, encoded
    return result

## 7. Checks: thinking control and the longest prompt

The thinking switch is part of Qwen's chat template: with thinking off there must be no
reasoning, with low reasoning there must be some. Then one token from the longest prompt shows
the largest prefill fits on the GPU.

In [ ]:
for condition in CONDITIONS:
    check = generate_completion([{"role": "user", "content": "What is 17 + 25? Return only the integer."}],
                                condition, max_new_tokens=1024)
    print(condition, {"text": check["text"], "reasoning_tokens": check["usage"]["completion_tokens_details"]["reasoning_tokens"],
                      "tokens": check["usage"]["completion_tokens"], "finish": check["finish_reason"]})
    if not check["text"].strip():
        raise RuntimeError(f"{condition}: no visible answer")
    if condition == "none" and check["reasoning"].strip():
        raise RuntimeError("Thinking disabled, but the model returned reasoning")
    if condition != "none" and not check["reasoning"].strip():
        raise RuntimeError(f"{condition}: no reasoning returned")

In [ ]:
longest_check = generate_completion(LONGEST_PROMPT["messages"], CONDITION, max_new_tokens=1)
print({"dataset": LONGEST_PROMPT["dataset"], "id": LONGEST_PROMPT["id"],
       "prompt_tokens": longest_check["usage"]["prompt_tokens"],
       "seconds": round(longest_check["latency_seconds"], 1),
       "peak_gpu_memory_gib": round(longest_check["peak_gpu_memory_gib"], 1)})

## 8. Runner

`run()` writes what `scripts/run_predictions.py` writes: one JSON line per request in
`OUT_DIR/RUN.jsonl`, with a manifest whose configuration a resumed run must match exactly. It
reuses the runner's prompt loader and its recovery of a line cut off by an interruption.
`report()` checks what the repository's evaluation will need: answers that parse, and answer
log-probabilities that align with the text.

In [ ]:
def request_parameters(condition):
    reasoning = {"effort": "none", "enabled": False} if condition == "none" else {"effort": condition}
    return {"max_tokens": MAX_NEW_TOKENS, "temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
            "min_p": MIN_P, "repetition_penalty": REPETITION_PENALTY, "seed": SEED, "logprobs": True,
            "reasoning": reasoning, "chat_template_kwargs": template_kwargs(condition)}


def configuration(paths, condition, pilot_size):
    config = {"provider": PROVIDER, "model": MODEL["model_id"], "parameters": request_parameters(condition),
              "backend": {"revision": MODEL["revision"], "precision": MODEL["precision"], "attention": "sdpa",
                          "linear_attention": LINEAR_ATTENTION_BACKENDS, "prefill_chunk_size": PREFILL_CHUNK_SIZE,
                          "max_total_tokens": MAX_TOTAL_TOKENS, "packages": PACKAGE_VERSIONS,
                          "logprobs": "selected token under the raw model logits, before temperature and filters"},
              "repetitions": 1, "limit_per_dataset": pilot_size,
              "prompt_files": {path.name: sha256_file(path) for path in paths}}
    if pilot_size:
        config["selection"] = "evenly spaced over each prompt file"
    return config


def evenly_spaced(prompts, count):
    """count prompts per dataset, spread over each file, so short and long programs both appear."""
    chosen = []
    for name in DATASETS:
        items = [prompt for prompt in prompts if prompt["dataset"] == name]
        picks = sorted({round(i * (len(items) - 1) / max(count - 1, 1)) for i in range(min(count, len(items)))})
        chosen += [items[i] for i in picks]
    return chosen


def complete(prompt, condition):
    """One request: the runner-format result, and the token-level record (None on error)."""
    try:
        result = generate_completion(prompt["messages"], condition)
    except Exception as error:  # e.g. out of GPU memory: saved as an error, retried with RETRY_ERRORS
        release_memory()
        return {"status": "error", "error": f"{type(error).__name__}: {error}"}, None
    tokens = {"token_ids": result.pop("token_ids"), "token_logprobs": result.pop("token_logprobs")}
    status = "completed" if result["finish_reason"] == "stop" and result["text"].strip() else "incomplete"
    return {"status": status, **result, "returned_model": f"{MODEL['model_id']}@{MODEL['revision']}",
            "backend_provider": PROVIDER}, tokens


def run(run_name, prompt_dir, out_dir, condition, pilot_size=None):
    """Run every prompt in prompt_dir (or pilot_size per dataset) and return the run log's path."""
    paths = prompt_paths(prompt_dir)
    prompts = load_prompts(paths, None)
    if pilot_size:
        prompts = evenly_spaced(prompts, pilot_size)
    parameters = request_parameters(condition)
    config = configuration(paths, condition, pilot_size)
    out_dir.mkdir(parents=True, exist_ok=True)
    log, manifest = out_dir / f"{run_name}.jsonl", out_dir / f"{run_name}.manifest.json"
    token_log = EXTRA / "tokens" / f"{run_name}.tokens.jsonl"
    token_log.parent.mkdir(parents=True, exist_ok=True)
    if manifest.exists():
        if json.loads(manifest.read_text(encoding="utf-8"))["configuration"] != config:
            raise RuntimeError("Run configuration changed; choose a new run name")
    elif log.exists():
        raise RuntimeError("Existing result log has no manifest")
    else:
        manifest.write_text(json.dumps({"created_at": now(), "configuration": config, "planned_requests": len(prompts)},
                                       indent=2) + "\n", encoding="utf-8")
    latest, attempts = {}, Counter()
    for row in read_results(log):
        key = (row["dataset"], row["id"], row["repetition"])
        latest[key] = row
        attempts[key] += 1
    pending = [prompt for prompt in prompts if (prompt["dataset"], prompt["id"], 0) not in latest
               or (RETRY_ERRORS and latest[(prompt["dataset"], prompt["id"], 0)]["status"] == "error")]
    print(f"{run_name}: {len(prompts) - len(pending)} saved, {len(pending)} to run", flush=True)
    statuses = Counter()
    with log.open("a", encoding="utf-8", newline="\n") as stream, \
            token_log.open("a", encoding="utf-8", newline="\n") as token_stream:
        for done, prompt in enumerate(pending, start=1):
            key = (prompt["dataset"], prompt["id"], 0)
            result, tokens = complete(prompt, condition)
            record = {"dataset": prompt["dataset"], "id": prompt["id"], "repetition": 0, "attempt": attempts[key] + 1,
                      "prompt_sha256": prompt["prompt_sha256"], "provider": PROVIDER,
                      "requested_model": MODEL["model_id"], "request_parameters": parameters, "saved_at": now(),
                      **result}
            stream.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
            stream.flush()
            os.fsync(stream.fileno())
            if tokens:
                token_stream.write(json.dumps({**{field: record[field] for field in ("dataset", "id", "repetition", "attempt")},
                                               **tokens}) + "\n")
                token_stream.flush()
            statuses[result["status"]] += 1
            if done == 1 or done % 10 == 0 or done == len(pending):
                usage = result.get("usage") or {}
                print(f"  {done}/{len(pending)} {prompt['dataset']} {prompt['id']}: {result['status']}, "
                      f"{usage.get('completion_tokens', 0)} tokens in {result.get('latency_seconds', 0):.0f}s", flush=True)
    print(dict(statuses))
    return log


def parses(parse, text):
    try:
        parse(text or "")
        return True
    except ValueError:
        return False


def report(log, verification=False):
    """Per dataset: statuses, token counts, speed, and whether the answers parse and their
    log-probabilities align with the text, as the repository's evaluation needs."""
    _, latest = load_run(log)
    for name in DATASETS:
        rows = [row for (dataset, _, _), row in latest.items() if dataset == name]
        if not rows:
            continue
        done = [row for row in rows if row["status"] == "completed"]
        usage = [row["usage"] for row in rows if row.get("usage")]
        line = {"items": len(rows), "statuses": dict(Counter(row["status"] for row in rows)),
                "finish": dict(Counter(row.get("finish_reason") for row in rows)),
                "median reasoning tokens": median(u["completion_tokens_details"]["reasoning_tokens"] for u in usage) if usage else None,
                "max generated tokens": max((u["completion_tokens"] for u in usage), default=None),
                "mean seconds": round(fmean(row["latency_seconds"] for row in rows if row.get("latency_seconds")), 1) if usage else None}
        if verification:
            judged = [row for row in done if parses(parse_verdict, row["text"]) and parses(parse_confidence, row["text"])]
            line["verdict and confidence parse"] = f"{len(judged)}/{len(done)}"
            line["verdict probability aligns"] = (f"{sum(verdict_probability(row['text'], row['logprobs']) is not None for row in judged)}"
                                                  f"/{len(judged)}")
            line["approved"] = f"{sum(parse_verdict(row['text']) == 'correct' for row in judged)}/{len(judged)}"
        else:
            answered = [row for row in done if parses(parse_prediction, row["text"])]
            line["answers parse"] = f"{len(answered)}/{len(done)}"
            line["answer log-probabilities align"] = (f"{sum(prediction_logprobs(row['text'], row['logprobs']) is not None for row in answered)}"
                                                      f"/{len(answered)}")
        print(name, json.dumps(line, indent=1))


def projected_hours(log, counts):
    """Hours for counts items per dataset, at the log's mean seconds per item."""
    _, latest = load_run(log)
    total = 0
    for name, count in counts.items():
        seconds = [row["latency_seconds"] for (dataset, _, _), row in latest.items()
                   if dataset == name and row.get("latency_seconds")]
        total += fmean(seconds) * count if seconds else 0
    return total / 3600

## 9. Pilot

Ten prompts per dataset, spread over each file, then their self-verification, as separate runs
under `_local/qwen38_27b/pilot/`.

In [ ]:
if not ADDITIONS:
    PILOT_PREDICTIONS = run(RUN_NAME + "_pilot", PROMPT_DIR, PILOT / "predictions", CONDITION, pilot_size=PILOT_SIZE)
    pilot_prompts = PILOT / "prompts" / "verification" / PILOT_PREDICTIONS.stem
    generate_verification_prompts(PILOT_PREDICTIONS, pilot_prompts)
    PILOT_VERIFICATIONS = run(PILOT_PREDICTIONS.stem + "_self_verification", pilot_prompts, PILOT / "verifications", CONDITION)

In [ ]:
if not ADDITIONS:
    report(PILOT_PREDICTIONS)
    report(PILOT_VERIFICATIONS, verification=True)
    prediction_hours = projected_hours(PILOT_PREDICTIONS, EXPECTED_COUNTS)
    verification_hours = projected_hours(PILOT_VERIFICATIONS, EXPECTED_COUNTS)
    print(f"Projected: {prediction_hours:.0f} h predictions + {verification_hours:.0f} h self-verification "
          f"= {prediction_hours + verification_hours:.0f} h")

## 10. Full prediction run

Only unlock this after the pilot shows:

1. no errors, and few truncations (`finish: length`);
2. all answers parse and all answer log-probabilities align;
3. low-reasoning rows have reasoning tokens;
4. verdicts and confidences parse and verdict probabilities align;
5. a projected time that finishes well before the paper deadline.

RE2 runs first. Rerunning this cell resumes the run.

In [ ]:
CONFIRM_FULL_RUN = False
assert CONFIRM_FULL_RUN or ADDITIONS, "Check the pilot report, then set CONFIRM_FULL_RUN = True"
PREDICTIONS = run(RUN_NAME, PROMPT_DIR, DATA / "predictions", CONDITION)
report(PREDICTIONS)

## 11. Self-verification

Once every prompt has a saved response, the verification prompts are written with the
repository's own script (one per answer; truncated responses and answers without a prediction
tag are skipped) and judged with the same model and settings. Rerunning this cell resumes.

In [ ]:
PREDICTIONS = DATA / "predictions" / f"{RUN_NAME}.jsonl"
_, latest = load_run(PREDICTIONS)
missing = len(PROMPTS) - len(latest)
errors = sum(row["status"] == "error" for row in latest.values())
if missing or errors:
    raise RuntimeError(f"{missing} prompts not run and {errors} failed requests; rerun the prediction cell "
                       "(with RETRY_ERRORS = True to retry the failures)")
VERIFICATION_PROMPTS = DATA / "prompts" / "verification" / RUN_NAME
for record in generate_verification_prompts(PREDICTIONS, VERIFICATION_PROMPTS).values():
    print(record["dataset"], record["count"], "verification prompts; skipped", record["skipped"])
audit(load_prompts(prompt_paths(VERIFICATION_PROMPTS), None), CONDITION)
VERIFICATIONS = run(VERIFICATION_RUN, VERIFICATION_PROMPTS, DATA / "verifications", CONDITION)
report(VERIFICATIONS, verification=True)

## 12. Results zip

Packs whatever exists so far, so it also works for sending back a pilot or a partial run.
Unzip it at the repository root.

In [ ]:
RESULTS = BUNDLE_ROOT / f"{RUN_NAME}_results.zip"
members = [*sorted((DATA / "predictions").glob(f"{RUN_NAME}.*")),
           *sorted((DATA / "prompts" / "verification" / RUN_NAME).glob("*")),
           *sorted((DATA / "verifications").glob(f"{VERIFICATION_RUN}.*")),
           *sorted(path for path in EXTRA.rglob("*") if path.is_file())]
with zipfile.ZipFile(RESULTS, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in members:
        archive.write(path, path.relative_to(BUNDLE_ROOT).as_posix())
        print(path.relative_to(BUNDLE_ROOT).as_posix(), path.stat().st_size, "bytes")
print("Created", RESULTS.name, RESULTS.stat().st_size, "bytes, sha256", sha256_file(RESULTS))

## 13. Release GPU memory

Closing or restarting the kernel is the most reliable way to release all CUDA memory. The
following cell removes the Python references first.

In [ ]:
del MODEL_INSTANCE
gc.collect()
torch.cuda.empty_cache()
print("Model reference removed. Restart the kernel to release the CUDA context.")